 This notebook converts Remy's original ORN and KC distances into long-form tables with odor abbreviations for
easy readability, and saves them to `manuscript/data/figure-04/04cde/neural_remy`.

Input files:
- `manuscript/by_imaging_panel/megamat17` (distances for 4 megamat17 flies)
    - `orn_terminals/xrda_orn_stim_rdm_concat_mm.nc`: ORN distances (9 flies)
    - `kc_soma_nls/xrds_stim_rdm_concat_mm.nc`: KC distances (4 flies, newer data only)
- `manuscript/by_imaging_panel/megamat_new_and_old_by_fly_17` (distances for new and old megamat17 MB flies)
    - `kc_soma_nls/xrds_stim_rdm_concat_mm.nc`

<u>Output files</u>

The following output files are saved in the directory `OdorSpaceShare/manuscript/data/figure-04/04cde/neural_remy`:
-  `tidy_orn_dists_abbrev.(pkl, .tsv)`: Long-form ORN distances with columns `['orn_remy', 'orn_remy_scaled']` and odor
abbrev. pairs in the index (189 columns)
- `tidy_orn_dists_inchi.(pkl, tsv)`: Save as `tidy_orn_dists_abbrev`, but with InChI identifiers in the index
- `tidy_kc_dists_abbrev.(pkl, tsv)`: Long-form KC distances with columns `['kc_remy', 'kc_remy_scaled',
'kc_remy_combo', 'kc_remy_combo_scaled']` and
odor abbrev. pairs in the index (189 columns)
- `tidy_kc_dists_inchi.(pkl, tsv)`: Same as `tidy_kc_dists_abbrev`, but with InChI identifiers in the index.

Descriptions of columns:
- `orn_remy`:

In [91]:
import json
from itertools import combinations
from pathlib import Path
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

kc_ord = ['2h', 'IaA', 'pa',
          '2-but', 'eb', 'ep',
          'aa', 'va',
          'B-cit', 'Lin',
          '6al', 't2h',
          '1-8ol', '1-5ol', '1-6ol',
          'benz', 'ms'
          ]
# ['uniform_dat', 'hemibrain_dat', 'uniform_emb', 'hemibrain_emb']

label_map = {
    'chem_rdkit': 'rdkit',
    'chem_fcfp': 'fcfp',
    'chem_ecfp': 'ecfp',
    'chem_pattern': 'pattern',
    'vcf_dat': 'VCF distance',
    'vcf_emb': 'VCF hyp. distance',
    'orn_dat': 'ORN distance',
    'orn_emb': 'ORN distance (embedded)',
    'orn_remy': 'ORN distance (remy)',
    'kc_dat': 'KC distance',
    'kc_emb': 'KC distance (embedded)',
    'kc_remy': 'KC distance (remy)',
    'uniform_dat': "modeled KCs (uniform)",
    'uniform_emb': "modeled KCs (uniform, emb.)",
    'hemibrain_dat': "modeled KCs (hemibrain)",
    'hemibrain_emb': "modeled KCs (hemibrain, emb.)",
    }

data_folder = Path("/home/remy/PycharmProjects/OdorSpaceShare/manuscript/data/figure-04/04cde")

# Load inchi --> abbrev map
with open(data_folder.joinpath('anoop_inchi_2_abbrev.json'), 'r') as f:
    anoop_inchi_2_abbrev = json.load(f)

# Also Make abbrev --> inchi map
anoop_abbrev_2_inchi = {v: k for k, v in anoop_inchi_2_abbrev.items()}

anoop_inchis = list(anoop_inchi_2_abbrev.keys())
inchi_pairs = list(combinations(anoop_inchis, 2))
abbrev_pairs = list(combinations(kc_ord, 2))


In [18]:
def preprocess_remy_distance_dataframe(df_distance):
    df_new = df_distance.rename(index=lambda x: x.split(" @ ")[0],
                                columns=lambda x: x.split(" @ ")[0]
                                )
    df_new = df_new.rename_axis(None, axis=0).rename_axis(None, axis=1)
    return df_new

# ORN distances

## Load Remy's ORN distances

For `panel_name='megamat17'`, make the following dataframes for ORN terminals:
- df_stim_rdm_orn_mean: column name `'orn_remy'`
- df_stim_rdm_orn_mean_scaled: column name `'orn_remy_mean_scaled'`
- df_stim_rdm_orn_scaled_mean: column name `'orn_remy_scaled_mean'`

In [94]:
panel_name = 'megamat17'

da_stim_rdm_concat_mm_orn = (xr.load_dataarray
                           (f"/home/remy/PycharmProjects/OdorSpaceShare/manuscript/data/by_imaging_panel"
                                        f"/{panel_name}/"
                                        "orn_terminals/xrda_orn_stim_rdm_concat_mm.nc"))

da_stim_rdm_concat_mm_orn = (da_stim_rdm_concat_mm_orn
                         .set_index(acq=['date_imaged', 'fly_num'])
                         # .set_index(rdms=['cell_mask_coord', 'metric'])
                         )

# compute "raw" mean correlation distances
da_stim_rdm_orn = (da_stim_rdm_concat_mm_orn.sel(metric='correlation'))
da_stim_rdm_orn_mean = da_stim_rdm_orn.mean(dim='acq')
df_stim_rdm_orn_mean = da_stim_rdm_orn_mean.to_pandas()

# compute scaled distances (mean, then scaled)
da_stim_rdm_orn_mean_scaled = (2.0 / da_stim_rdm_orn_mean.max(dim=['stim_row', 'stim_col'])) * da_stim_rdm_orn_mean
df_stim_rdm_orn_mean_scaled = da_stim_rdm_orn_mean_scaled.to_pandas()
# display(df_stim_rdm_orn_mean_scaled)

# compute scaled distances (scaled, then mean)
da_stim_rdm_orn_scaled = (2.0 / da_stim_rdm_orn.max(dim=['stim_row', 'stim_col'])) * da_stim_rdm_orn
da_stim_rdm_orn_scaled_mean = da_stim_rdm_orn_scaled.mean(dim='acq')
df_stim_rdm_orn_scaled_mean = da_stim_rdm_orn_scaled_mean.to_pandas()
display(df_stim_rdm_orn_scaled_mean)

orn_neural_dists = dict(
        orn_remy=df_stim_rdm_orn_mean,
        orn_remy_mean_scaled=df_stim_rdm_orn_mean_scaled,
        orn_remy_scaled_mean=df_stim_rdm_orn_scaled_mean,
        )

print('megamat17 (9 flies only): `orn_remy`')
display(orn_neural_dists['orn_remy'])

print('megamat17 (4 flies only): `orn_remy_mean_scaled`')
display(orn_neural_dists['orn_remy_mean_scaled'])

print('megamat17 (4 flies only): `orn_remy_scaled_mean`')
display(orn_neural_dists['orn_remy_scaled_mean'])

stim_col,1-5ol @ -3.0,1-6ol @ -3.0,1-8ol @ -3.0,2-but @ -3.0,2h @ -3.0,6al @ -3.0,B-cit @ -3.0,IaA @ -3.0,Lin @ -3.0,aa @ -3.0,benz @ -3.0,eb @ -3.0,ep @ -3.0,ms @ -3.0,pa @ -3.0,t2h @ -3.0,va @ -3.0
stim_row,,,,,,,,,,,,,,,,,
1-5ol @ -3.0,0.000000,0.375549,0.724386,0.828608,0.701923,0.605481,1.289033,0.865809,1.381301,1.412166,1.394403,0.640916,0.654050,1.576222,0.750748,0.745758,1.292177
1-6ol @ -3.0,0.375549,0.000000,0.699511,1.071963,0.693697,0.711184,1.098661,0.750946,1.117493,1.483642,1.496369,0.665082,0.769182,1.640945,0.635269,0.805640,1.469973
1-8ol @ -3.0,0.724386,0.699511,0.000000,1.195706,0.930877,0.916047,0.976079,1.270717,1.001895,1.640719,1.747800,0.921453,1.007523,1.737956,1.108931,0.862057,1.394749
2-but @ -3.0,0.828608,1.071963,1.195706,0.000000,0.943714,0.938014,1.020053,1.211751,1.637684,1.221255,1.510317,0.627830,0.450698,1.256401,1.319103,1.043522,1.176157
2h @ -3.0,0.701923,0.693697,0.930877,0.943714,0.000000,0.864437,1.158510,0.479455,1.064747,1.282436,1.580273,0.707181,0.814154,1.715160,0.352105,0.980699,1.140276
6al @ -3.0,0.605481,0.711184,0.916047,0.938014,0.864437,0.000000,1.059802,1.313482,1.282994,1.258443,1.297252,1.038136,1.072485,1.585253,1.179447,0.338547,0.884068
B-cit @ -3.0,1.289033,1.098661,0.976079,1.020053,1.158510,1.059802,0.000000,1.248061,1.089210,1.239720,1.405788,1.089089,1.001092,1.336312,1.448012,0.971697,1.485610
IaA @ -3.0,0.865809,0.750946,1.270717,1.211751,0.479455,1.313482,1.248061,0.000000,1.377238,1.582910,1.517606,0.710116,0.808972,1.561872,0.361220,1.437915,1.452749
Lin @ -3.0,1.381301,1.117493,1.001895,1.637684,1.064747,1.282994,1.089210,1.377238,0.000000,1.426219,1.694102,1.649857,1.741785,1.713920,1.305090,1.060322,1.412421


megamat17 (9 flies only): `orn_remy`


stim_col,1-5ol @ -3.0,1-6ol @ -3.0,1-8ol @ -3.0,2-but @ -3.0,2h @ -3.0,6al @ -3.0,B-cit @ -3.0,IaA @ -3.0,Lin @ -3.0,aa @ -3.0,benz @ -3.0,eb @ -3.0,ep @ -3.0,ms @ -3.0,pa @ -3.0,t2h @ -3.0,va @ -3.0
stim_row,,,,,,,,,,,,,,,,,
1-5ol @ -3.0,0.000000,0.223439,0.429620,0.498254,0.421255,0.362927,0.775826,0.520031,0.827723,0.875689,0.840561,0.386791,0.393776,0.951336,0.449823,0.447456,0.798579
1-6ol @ -3.0,0.223439,0.000000,0.419934,0.643171,0.417559,0.426926,0.662061,0.451292,0.672271,0.922147,0.902641,0.397104,0.458814,0.988172,0.382417,0.482146,0.902255
1-8ol @ -3.0,0.429620,0.419934,0.000000,0.717508,0.560798,0.550018,0.586691,0.764759,0.603709,1.016351,1.048579,0.550588,0.600523,1.047605,0.667714,0.520386,0.861899
2-but @ -3.0,0.498254,0.643171,0.717508,0.000000,0.566020,0.563865,0.612002,0.727954,0.982736,0.749830,0.909349,0.376073,0.268752,0.751620,0.793270,0.625406,0.725107
2h @ -3.0,0.421255,0.417559,0.560798,0.566020,0.000000,0.518601,0.698844,0.289735,0.642232,0.793890,0.953129,0.427971,0.491370,1.031932,0.211053,0.586664,0.701311
6al @ -3.0,0.362927,0.426926,0.550018,0.563865,0.518601,0.000000,0.640134,0.789644,0.770932,0.778893,0.784422,0.624465,0.644044,0.954482,0.708256,0.203800,0.547210
B-cit @ -3.0,0.775826,0.662061,0.586691,0.612002,0.698844,0.640134,0.000000,0.752336,0.657371,0.765042,0.847643,0.655843,0.600425,0.802708,0.873627,0.584890,0.916365
IaA @ -3.0,0.520031,0.451292,0.764759,0.727954,0.289735,0.789644,0.752336,0.000000,0.826738,0.977844,0.912429,0.429206,0.487503,0.940014,0.219633,0.862226,0.891023
Lin @ -3.0,0.827723,0.672271,0.603709,0.982736,0.642232,0.770932,0.657371,0.826738,0.000000,0.888690,1.020607,0.989784,1.045773,1.031770,0.784647,0.636398,0.871642


megamat17 (4 flies only): `orn_remy_mean_scaled`


stim_col,1-5ol @ -3.0,1-6ol @ -3.0,1-8ol @ -3.0,2-but @ -3.0,2h @ -3.0,6al @ -3.0,B-cit @ -3.0,IaA @ -3.0,Lin @ -3.0,aa @ -3.0,benz @ -3.0,eb @ -3.0,ep @ -3.0,ms @ -3.0,pa @ -3.0,t2h @ -3.0,va @ -3.0
stim_row,,,,,,,,,,,,,,,,,
1-5ol @ -3.0,0.000000,0.412306,0.792765,0.919413,0.777329,0.669699,1.431608,0.959597,1.527372,1.615883,1.551062,0.713733,0.726622,1.755471,0.830045,0.825677,1.473593
1-6ol @ -3.0,0.412306,0.000000,0.774891,1.186824,0.770509,0.787793,1.221682,0.832756,1.240522,1.701611,1.665617,0.732764,0.846635,1.823444,0.705663,0.889689,1.664904
1-8ol @ -3.0,0.792765,0.774891,0.000000,1.323996,1.034824,1.014931,1.082603,1.411187,1.114006,1.875442,1.934911,1.015983,1.108128,1.933115,1.232112,0.960254,1.590437
2-but @ -3.0,0.919413,1.186824,1.323996,0.000000,1.044459,1.040484,1.129310,1.343271,1.813413,1.383639,1.677994,0.693955,0.495920,1.386942,1.463798,1.154043,1.338018
2h @ -3.0,0.777329,0.770509,1.034824,1.044459,0.000000,0.956958,1.289557,0.534639,1.185091,1.464941,1.758780,0.789723,0.906710,1.904193,0.389449,1.082553,1.294109
6al @ -3.0,0.669699,0.787793,1.014931,1.040484,0.956958,0.000000,1.181220,1.457107,1.422578,1.437267,1.447471,1.152306,1.188435,1.761276,1.306923,0.376066,1.009749
B-cit @ -3.0,1.431608,1.221682,1.082603,1.129310,1.289557,1.181220,0.000000,1.388263,1.213027,1.411709,1.564130,1.210208,1.107946,1.481213,1.612077,1.079281,1.690940
IaA @ -3.0,0.959597,0.832756,1.411187,1.343271,0.534639,1.457107,1.388263,0.000000,1.525555,1.804386,1.683678,0.792001,0.899575,1.734580,0.405282,1.591040,1.644177
Lin @ -3.0,1.527372,1.240522,1.114006,1.813413,1.185091,1.422578,1.213027,1.525555,0.000000,1.639872,1.883295,1.826418,1.929733,1.903894,1.447886,1.174326,1.608415


megamat17 (4 flies only): `orn_remy_scaled_mean`


stim_col,1-5ol @ -3.0,1-6ol @ -3.0,1-8ol @ -3.0,2-but @ -3.0,2h @ -3.0,6al @ -3.0,B-cit @ -3.0,IaA @ -3.0,Lin @ -3.0,aa @ -3.0,benz @ -3.0,eb @ -3.0,ep @ -3.0,ms @ -3.0,pa @ -3.0,t2h @ -3.0,va @ -3.0
stim_row,,,,,,,,,,,,,,,,,
1-5ol @ -3.0,0.000000,0.375549,0.724386,0.828608,0.701923,0.605481,1.289033,0.865809,1.381301,1.412166,1.394403,0.640916,0.654050,1.576222,0.750748,0.745758,1.292177
1-6ol @ -3.0,0.375549,0.000000,0.699511,1.071963,0.693697,0.711184,1.098661,0.750946,1.117493,1.483642,1.496369,0.665082,0.769182,1.640945,0.635269,0.805640,1.469973
1-8ol @ -3.0,0.724386,0.699511,0.000000,1.195706,0.930877,0.916047,0.976079,1.270717,1.001895,1.640719,1.747800,0.921453,1.007523,1.737956,1.108931,0.862057,1.394749
2-but @ -3.0,0.828608,1.071963,1.195706,0.000000,0.943714,0.938014,1.020053,1.211751,1.637684,1.221255,1.510317,0.627830,0.450698,1.256401,1.319103,1.043522,1.176157
2h @ -3.0,0.701923,0.693697,0.930877,0.943714,0.000000,0.864437,1.158510,0.479455,1.064747,1.282436,1.580273,0.707181,0.814154,1.715160,0.352105,0.980699,1.140276
6al @ -3.0,0.605481,0.711184,0.916047,0.938014,0.864437,0.000000,1.059802,1.313482,1.282994,1.258443,1.297252,1.038136,1.072485,1.585253,1.179447,0.338547,0.884068
B-cit @ -3.0,1.289033,1.098661,0.976079,1.020053,1.158510,1.059802,0.000000,1.248061,1.089210,1.239720,1.405788,1.089089,1.001092,1.336312,1.448012,0.971697,1.485610
IaA @ -3.0,0.865809,0.750946,1.270717,1.211751,0.479455,1.313482,1.248061,0.000000,1.377238,1.582910,1.517606,0.710116,0.808972,1.561872,0.361220,1.437915,1.452749
Lin @ -3.0,1.381301,1.117493,1.001895,1.637684,1.064747,1.282994,1.089210,1.377238,0.000000,1.426219,1.694102,1.649857,1.741785,1.713920,1.305090,1.060322,1.412421


## Combine and save ORN distances

In [96]:
tidy_orn_dists_abbrev = pd.concat([v.stack().rename(k) for k, v in orn_neural_dists.items()], axis=1)
tidy_orn_dists_abbrev = tidy_orn_dists_abbrev.rename(index=lambda x: x.split(" @ ")[0]).rename_axis(index=[None, None])
display(tidy_orn_dists_abbrev)

tidy_orn_dists_inchi = tidy_orn_dists_abbrev.rename(index=anoop_abbrev_2_inchi)
display(tidy_orn_dists_inchi)

# Save to data/manuscript/figure-04/04cde/neural_remy
tidy_orn_dists_inchi.to_pickle(data_folder.joinpath('neural_remy', 'tidy_orn_dists_inchi.pkl'))
tidy_orn_dists_inchi.to_csv(data_folder.joinpath('neural_remy', 'tidy_orn_dists_inchi.tsv'), sep='\t')
print("`tidy_orn_dists_inchi` saved.")

tidy_orn_dists_abbrev.to_pickle(data_folder.joinpath('neural_remy', 'tidy_orn_dists_abbrev.pkl'))
tidy_orn_dists_abbrev.to_csv(data_folder.joinpath('neural_remy', 'tidy_orn_dists_abbrev.tsv'), sep='\t')
print("`tidy_kc_dists_abbrev` saved.")

orn_remy  orn_remy_mean_scaled  orn_remy_scaled_mean
1-5ol 1-5ol  0.000000              0.000000              0.000000
      1-6ol  0.223439              0.412306              0.375549
      1-8ol  0.429620              0.792765              0.724386
      2-but  0.498254              0.919413              0.828608
      2h     0.421255              0.777329              0.701923
...               ...                   ...                   ...
va    ep     0.717029              1.323112              1.164675
      ms     0.962242              1.775597              1.559601
      pa     0.905194              1.670327              1.478217
      t2h    0.704344              1.299705              1.137801
      va     0.000000              0.000000              0.000000

[289 rows x 3 columns]

orn_remy  \
InChI=1S/C5H12O/c1-2-3-4-5-6/h6H,2-5H2,1H3         InChI=1S/C5H12O/c1-2-3-4-5-6/h6H,2-5H2,1H3          0.000000   
                                                   InChI=1S/C6H14O/c1-2-3-4-5-6-7/h7H,2-6H2,1H3        0.223439   
                                                   InChI=1S/C8H18O/c1-2-3-4-5-6-7-8-9/h9H,2-8H2,1H3    0.429620   
                                                   InChI=1S/C4H8O/c1-3-4(2)5/h3H2,1-2H3                0.498254   
                                                   InChI=1S/C7H14O/c1-3-4-5-6-7(2)8/h3-6H2,1-2H3       0.421255   
...                                                                                                         ...   
InChI=1S/C5H10O2/c1-2-3-4-5(6)7/h2-4H2,1H3,(H,6,7) InChI=1S/C5H10O2/c1-3-5(6)7-4-2/h3-4H2,1-2H3        0.717029   
                                                   InChI=1S/C8H8O3/c1-11-8(10)6-4-2-3-5-7(6)9/h2-5...  0.962242   
                                                   InChI=1S/C7H14O2/c1-3-4-5-6-9-7(2)8/h3-6H2,1-2H3    0.905194   
                                                   InChI=1S/C6H10O/c1-2-3-4-5-6-7/h4-6H,2-3H2,1H3/...  0.704344   
                                                   InChI=1S/C5H10O2/c1-2-3-4-5(6)7/h2-4H2,1H3,(H,6,7)  0.000000   

                                                                                                       orn_remy_mean_scaled  \
InChI=1S/C5H12O/c1-2-3-4-5-6/h6H,2-5H2,1H3         InChI=1S/C5H12O/c1-2-3-4-5-6/h6H,2-5H2,1H3                      0.000000   
                                                   InChI=1S/C6H14O/c1-2-3-4-5-6-7/h7H,2-6H2,1H3                    0.412306   
                                                   InChI=1S/C8H18O/c1-2-3-4-5-6-7-8-9/h9H,2-8H2,1H3                0.792765   
                                                   InChI=1S/C4H8O/c1-3-4(2)5/h3H2,1-2H3                            0.919413   
                                                   InChI=1S/C7H14O/c1-3-4-5-6-7(2)8/h3-6H2,1-2H3                   0.777329   
...                                                                                                                     ...   
InChI=1S/C5H10O2/c1-2-3-4-5(6)7/h2-4H2,1H3,(H,6,7) InChI=1S/C5H10O2/c1-3-5(6)7-4-2/h3-4H2,1-2H3                    1.323112   
                                                   InChI=1S/C8H8O3/c1-11-8(10)6-4-2-3-5-7(6)9/h2-5...              1.775597   
                                                   InChI=1S/C7H14O2/c1-3-4-5-6-9-7(2)8/h3-6H2,1-2H3                1.670327   
                                                   InChI=1S/C6H10O/c1-2-3-4-5-6-7/h4-6H,2-3H2,1H3/...              1.299705   
                                                   InChI=1S/C5H10O2/c1-2-3-4-5(6)7/h2-4H2,1H3,(H,6,7)              0.000000   

                                                                                                       orn_remy_scaled_mean  
InChI=1S/C5H12O/c1-2-3-4-5-6/h6H,2-5H2,1H3         InChI=1S/C5H12O/c1-2-3-4-5-6/h6H,2-5H2,1H3                      0.000000  
                                                   InChI=1S/C6H14O/c1-2-3-4-5-6-7/h7H,2-6H2,1H3                    0.375549  
                                                   InChI=1S/C8H18O/c1-2-3-4-5-6-7-8-9/h9H,2-8H2,1H3                0.724386  
                                                   InChI=1S/C4H8O/c1-3-4(2)5/h3H2,1-2H3                            0.828608  
                                                   InChI=1S/C7H14O/c1-3-4-5-6-7(2)8/h3-6H2,1-2H3                   0.701923  
...                                                                                                                     ...  
InChI=1S/C5H10O2/c1-2-3-4-5(6)7/h2-4H2,1H3,(H,6,7) InChI=1S/C5H10O2/c1-3-5(6)7-4-2/h3-4H2,1-2H3                    1.164675  
                                                   InChI=1S/C8H8O3/c1-11-8(10)6-4-2-3-5-7(6)9/h2-5...              1.559601  
                                                   InChI=1S/C7H

`tidy_orn_dists_inchi` saved.
`tidy_kc_dists_abbrev` saved.


# KC distances

For distances computed for megamat17 flies and the additional older flies, use `panel_name = 'megamat17'`.

Use the following setttings:
- `metric='correlation'`: Pearson correlation used as the distance metric.
- `cell_mask_coord='iscell_good_xid0'`: Odor distances computed from responding neuron clusters only. Clusters were
computed using `rastermap`, and the responder clusters were manually selected (non-responders or bad clusters were
dropped before the calculation.

For both groups of KC distances (`megamat17` and `megamat_new_and_old_by_fly_17`), we compute 3 versions of the
distance. In addition to the "raw" unscaled distances (`kc_remy` and `kc_remy_combo`) , we also compute scaled versions of odor
distances. The scaling factor used for a distance matrix $D$ is $$D_{scaled} = D * \frac{2.0}{max(D)}$$

For the scaled version, we compute two verions:
1. `kc_*_mean_scaled`: The distance matrices are averaged across flies. Then, the mean distance matrix is scaled.
2. `kc_*_scaled_mean`: Scaling is performed on a fly-by-fly basis first. The scaled distance matrices are then
averaged across flies.

<u>Output files</u>:

The long-form, tidy KC distances are saved in the folder `OdorSpaceShare/manuscript/data/figure-04/04cde/neural_remy`
 as the following files:
- tidy_kc_dists_abbrev(.pkl, .tsv)
- tidy_kc_dists_abbrev(.pkl, .tsv)

The columns names are as follows:
- For `megamat17`:
    1. kc_remy
    2. kc_remy_mean_scaled
    3. kc_remy_scaled_mean
- For `megamat17`:
    1. kc_remy_combo
    2. kc_remy_combo_mean_scaled
    3. kc_remy_combo_scaled_mean

All column names:
```python
columns = [
    'kc_remy',
    'kc_remy_mean_scaled',
    'kc_remy_scaled_mean',
    'kc_remy_combo',
    'kc_remy_combo_mean_scaled',
    'kc_remy_combo_scaled_mean'
]

```


## Load Remy's original megamat17 distances

For `panel_name='megamat17'`, make the following dataframes:
- df_stim_rdm_mean: column name `'kc_remy'`
- df_stim_rdm_mean_scaled: column name `'kc_remy_mean_scaled'`
- df_stim_rdm_scaled_mean: column name `'kc_remy_scaled_mean'`



In [77]:
panel_name = 'megamat17'

ds_stim_rdm_concat_mm = xr.load_dataset(f"/home/remy/PycharmProjects/OdorSpaceShare/manuscript/data/by_imaging_panel"
                                        f"/{panel_name}/"
                                        "kc_soma_nls/xrds_stim_rdm_concat_mm.nc")
ds_stim_rdm_concat_mm = (ds_stim_rdm_concat_mm
                         .set_index(acq=['date_imaged', 'fly_num', 'thorimage_name'])
                         .set_index(rdms=['cell_mask_coord', 'metric'])
                         )

# compute "raw" mean correlation distances
da_stim_rdm = (ds_stim_rdm_concat_mm['Fc_zscore'].sel(cell_mask_coord='iscell_good_xid0',
                                                      metric='correlation'))
da_stim_rdm_mean = da_stim_rdm.mean(dim='acq')
df_stim_rdm_mean = da_stim_rdm_mean.to_pandas()

# compute scaled distances (mean, then scaled)
da_stim_rdm_mean_scaled = (2.0 / da_stim_rdm_mean.max(dim=['stim_row', 'stim_col'])) * da_stim_rdm_mean
df_stim_rdm_mean_scaled = da_stim_rdm_mean_scaled.to_pandas()

# compute scaled distances (scaled, then mean)
da_stim_rdm_scaled = (2.0 / da_stim_rdm.max(dim=['stim_row', 'stim_col'])) * da_stim_rdm
da_stim_rdm_scaled_mean = da_stim_rdm_scaled.mean(dim='acq')
df_stim_rdm_scaled_mean = da_stim_rdm_scaled_mean.to_pandas()

neural_dists = dict(
        kc_remy=df_stim_rdm_mean,
        kc_remy_mean_scaled=df_stim_rdm_mean_scaled,
        kc_remy_scaled_mean=df_stim_rdm_scaled_mean,
        )

print('megamat17 (4 flies only): `kc_remy`')
display(neural_dists['kc_remy'])

print('megamat17 (4 flies only): `kc_remy_mean_scaled`')
display(neural_dists['kc_remy_mean_scaled'])

print('megamat17 (4 flies only): `kc_remy_scaled_mean`')
display(neural_dists['kc_remy_scaled_mean'])

megamat17 (4 flies only): `kc_remy`


stim_col,1-5ol @ -3.0,1-6ol @ -3.0,1-8ol @ -3.0,2-but @ -3.0,2h @ -3.0,6al @ -3.0,B-cit @ -3.0,IaA @ -3.0,Lin @ -3.0,aa @ -3.0,benz @ -3.0,eb @ -3.0,ep @ -3.0,ms @ -3.0,pa @ -3.0,t2h @ -3.0,va @ -3.0
stim_row,,,,,,,,,,,,,,,,,
1-5ol @ -3.0,0.000000,0.276908,0.560735,0.948798,0.946051,0.548509,0.943185,1.020103,1.032223,1.102268,0.802469,0.897195,0.894850,1.026726,0.950333,0.676765,1.102698
1-6ol @ -3.0,0.276908,0.000000,0.477404,1.006534,0.887367,0.655256,0.893388,1.035177,0.937968,1.111454,0.871559,0.978285,0.965034,1.042757,0.881169,0.736568,1.102740
1-8ol @ -3.0,0.560735,0.477404,0.000000,0.919628,0.975512,0.758980,0.637021,1.057172,0.786386,1.009675,0.967985,0.943105,0.916712,0.980727,0.967088,0.794486,1.027697
2-but @ -3.0,0.948798,1.006534,0.919628,0.000000,0.981587,0.944012,0.912828,1.065555,1.008158,0.959367,1.073330,0.757930,0.452500,0.935340,1.100119,1.028267,0.970156
2h @ -3.0,0.946051,0.887367,0.975512,0.981587,0.000000,0.854629,0.987053,0.559901,0.820452,0.926241,0.967598,0.897767,0.995038,1.057934,0.452557,0.941541,0.951862
6al @ -3.0,0.548509,0.655256,0.758980,0.944012,0.854629,0.000000,0.879522,1.094220,1.029333,0.918652,0.788592,0.998267,1.042828,1.066139,1.035511,0.392591,0.875284
B-cit @ -3.0,0.943185,0.893388,0.637021,0.912828,0.987053,0.879522,0.000000,1.035163,0.785000,0.749993,1.003655,0.992478,0.987756,0.939148,1.040234,0.954793,0.912620
IaA @ -3.0,1.020103,1.035177,1.057172,1.065555,0.559901,1.094220,1.035163,0.000000,0.908655,1.099979,1.036502,0.769348,0.909219,1.022390,0.343340,1.135268,1.119122
Lin @ -3.0,1.032223,0.937968,0.786386,1.008158,0.820452,1.029333,0.785000,0.908655,0.000000,0.920961,0.956638,1.023200,1.051312,0.964448,0.961260,0.995630,0.977186


megamat17 (4 flies only): `kc_remy_mean_scaled`


stim_col,1-5ol @ -3.0,1-6ol @ -3.0,1-8ol @ -3.0,2-but @ -3.0,2h @ -3.0,6al @ -3.0,B-cit @ -3.0,IaA @ -3.0,Lin @ -3.0,aa @ -3.0,benz @ -3.0,eb @ -3.0,ep @ -3.0,ms @ -3.0,pa @ -3.0,t2h @ -3.0,va @ -3.0
stim_row,,,,,,,,,,,,,,,,,
1-5ol @ -3.0,0.000000,0.487828,0.987846,1.671495,1.666657,0.966307,1.661608,1.797113,1.818466,1.941864,1.413709,1.580587,1.576456,1.808782,1.674201,1.192255,1.942621
1-6ol @ -3.0,0.487828,0.000000,0.841042,1.773209,1.563274,1.154363,1.573880,1.823670,1.652418,1.958047,1.535424,1.723444,1.700099,1.837024,1.552354,1.297611,1.942696
1-8ol @ -3.0,0.987846,0.841042,0.000000,1.620108,1.718557,1.337095,1.122240,1.862418,1.385376,1.778743,1.705297,1.661467,1.614969,1.727745,1.703718,1.399646,1.810492
2-but @ -3.0,1.671495,1.773209,1.620108,0.000000,1.729261,1.663065,1.608128,1.877186,1.776070,1.690116,1.890884,1.335243,0.797168,1.647788,1.938078,1.811497,1.709123
2h @ -3.0,1.666657,1.563274,1.718557,1.729261,0.000000,1.505598,1.738889,0.986377,1.445390,1.631757,1.704617,1.581594,1.752956,1.863761,0.797269,1.658711,1.676893
6al @ -3.0,0.966307,1.154363,1.337095,1.663065,1.505598,0.000000,1.549452,1.927686,1.813374,1.618388,1.389262,1.758645,1.837149,1.878215,1.824258,0.691628,1.541986
B-cit @ -3.0,1.661608,1.573880,1.122240,1.608128,1.738889,1.549452,0.000000,1.823645,1.382933,1.321261,1.768137,1.748446,1.740129,1.654496,1.832578,1.682058,1.607762
IaA @ -3.0,1.797113,1.823670,1.862418,1.877186,0.986377,1.927686,1.823645,0.000000,1.600776,1.937831,1.826004,1.355359,1.601770,1.801143,0.604861,2.000000,1.971555
Lin @ -3.0,1.818466,1.652418,1.385376,1.776070,1.445390,1.813374,1.382933,1.600776,0.000000,1.622456,1.685307,1.802570,1.852094,1.699067,1.693449,1.754000,1.721507


megamat17 (4 flies only): `kc_remy_scaled_mean`


stim_col,1-5ol @ -3.0,1-6ol @ -3.0,1-8ol @ -3.0,2-but @ -3.0,2h @ -3.0,6al @ -3.0,B-cit @ -3.0,IaA @ -3.0,Lin @ -3.0,aa @ -3.0,benz @ -3.0,eb @ -3.0,ep @ -3.0,ms @ -3.0,pa @ -3.0,t2h @ -3.0,va @ -3.0
stim_row,,,,,,,,,,,,,,,,,
1-5ol @ -3.0,0.000000,0.476571,0.969711,1.643964,1.636903,0.948734,1.632497,1.767234,1.785298,1.907185,1.388153,1.557113,1.551742,1.777486,1.646505,1.172430,1.908381
1-6ol @ -3.0,0.476571,0.000000,0.825337,1.744128,1.536635,1.133350,1.547494,1.792762,1.624845,1.922033,1.508721,1.695807,1.673437,1.804145,1.527290,1.274090,1.908305
1-8ol @ -3.0,0.969711,0.825337,0.000000,1.592060,1.687797,1.315354,1.105537,1.829213,1.361773,1.748559,1.674579,1.633928,1.587604,1.697260,1.673421,1.374702,1.778949
2-but @ -3.0,1.643964,1.744128,1.592060,0.000000,1.700025,1.632075,1.579457,1.846604,1.742769,1.661456,1.857108,1.316508,0.785731,1.621305,1.904712,1.778232,1.677901
2h @ -3.0,1.636903,1.536635,1.687797,1.700025,0.000000,1.477155,1.709869,0.972714,1.420590,1.603184,1.673533,1.556201,1.724243,1.830850,0.785012,1.627404,1.647638
6al @ -3.0,0.948734,1.133350,1.315354,1.632075,1.477155,0.000000,1.524111,1.894282,1.781017,1.591543,1.363047,1.727335,1.804653,1.844132,1.792537,0.680782,1.515718
B-cit @ -3.0,1.632497,1.547494,1.105537,1.579457,1.709869,1.524111,0.000000,1.791499,1.359638,1.298943,1.736063,1.719017,1.710316,1.624644,1.800755,1.651953,1.579109
IaA @ -3.0,1.767234,1.792762,1.829213,1.846604,0.972714,1.894282,1.791499,0.000000,1.571740,1.904022,1.793014,1.334608,1.576134,1.769253,0.596505,1.964159,1.937174
Lin @ -3.0,1.785298,1.624845,1.361773,1.742769,1.420590,1.781017,1.359638,1.571740,0.000000,1.593641,1.655534,1.770712,1.818085,1.667738,1.665266,1.722784,1.691831


In [19]:
neural_dists = {
    'kc_remy': da_stim_rdm.mean(dim='acq').to_pandas(),
    'kc_remy_scaled': da_stim_rdm_scaled_mean.to_pandas()
    }
neural_dists = {k: preprocess_remy_distance_dataframe(v) for k, v in neural_dists.items()}

for k, v in neural_dists.items():
    print(k)
    display(v)

kc_remy


,1-5ol,1-6ol,1-8ol,2-but,2h,6al,B-cit,IaA,Lin,aa,benz,eb,ep,ms,pa,t2h,va
1-5ol,0.000000,0.276908,0.560735,0.948798,0.946051,0.548509,0.943185,1.020103,1.032223,1.102268,0.802469,0.897195,0.894850,1.026726,0.950333,0.676765,1.102698
1-6ol,0.276908,0.000000,0.477404,1.006534,0.887367,0.655256,0.893388,1.035177,0.937968,1.111454,0.871559,0.978285,0.965034,1.042757,0.881169,0.736568,1.102740
1-8ol,0.560735,0.477404,0.000000,0.919628,0.975512,0.758980,0.637021,1.057172,0.786386,1.009675,0.967985,0.943105,0.916712,0.980727,0.967088,0.794486,1.027697
2-but,0.948798,1.006534,0.919628,0.000000,0.981587,0.944012,0.912828,1.065555,1.008158,0.959367,1.073330,0.757930,0.452500,0.935340,1.100119,1.028267,0.970156
2h,0.946051,0.887367,0.975512,0.981587,0.000000,0.854629,0.987053,0.559901,0.820452,0.926241,0.967598,0.897767,0.995038,1.057934,0.452557,0.941541,0.951862
6al,0.548509,0.655256,0.758980,0.944012,0.854629,0.000000,0.879522,1.094220,1.029333,0.918652,0.788592,0.998267,1.042828,1.066139,1.035511,0.392591,0.875284
B-cit,0.943185,0.893388,0.637021,0.912828,0.987053,0.879522,0.000000,1.035163,0.785000,0.749993,1.003655,0.992478,0.987756,0.939148,1.040234,0.954793,0.912620
IaA,1.020103,1.035177,1.057172,1.065555,0.559901,1.094220,1.035163,0.000000,0.908655,1.099979,1.036502,0.769348,0.909219,1.022390,0.343340,1.135268,1.119122
Lin,1.032223,0.937968,0.786386,1.008158,0.820452,1.029333,0.785000,0.908655,0.000000,0.920961,0.956638,1.023200,1.051312,0.964448,0.961260,0.995630,0.977186
aa,1.102268,1.111454,1.009675,0.959367,0.926241,0.918652,0.749993,1.099979,0.920961,0.000000,1.027681,1.078977,1.103095,0.993947,1.113135,1.027505,0.760938


kc_remy_scaled


,1-5ol,1-6ol,1-8ol,2-but,2h,6al,B-cit,IaA,Lin,aa,benz,eb,ep,ms,pa,t2h,va
1-5ol,0.000000,0.476571,0.969711,1.643964,1.636903,0.948734,1.632497,1.767234,1.785298,1.907185,1.388153,1.557113,1.551742,1.777486,1.646505,1.172430,1.908381
1-6ol,0.476571,0.000000,0.825337,1.744128,1.536635,1.133350,1.547494,1.792762,1.624845,1.922033,1.508721,1.695807,1.673437,1.804145,1.527290,1.274090,1.908305
1-8ol,0.969711,0.825337,0.000000,1.592060,1.687797,1.315354,1.105537,1.829213,1.361773,1.748559,1.674579,1.633928,1.587604,1.697260,1.673421,1.374702,1.778949
2-but,1.643964,1.744128,1.592060,0.000000,1.700025,1.632075,1.579457,1.846604,1.742769,1.661456,1.857108,1.316508,0.785731,1.621305,1.904712,1.778232,1.677901
2h,1.636903,1.536635,1.687797,1.700025,0.000000,1.477155,1.709869,0.972714,1.420590,1.603184,1.673533,1.556201,1.724243,1.830850,0.785012,1.627404,1.647638
6al,0.948734,1.133350,1.315354,1.632075,1.477155,0.000000,1.524111,1.894282,1.781017,1.591543,1.363047,1.727335,1.804653,1.844132,1.792537,0.680782,1.515718
B-cit,1.632497,1.547494,1.105537,1.579457,1.709869,1.524111,0.000000,1.791499,1.359638,1.298943,1.736063,1.719017,1.710316,1.624644,1.800755,1.651953,1.579109
IaA,1.767234,1.792762,1.829213,1.846604,0.972714,1.894282,1.791499,0.000000,1.571740,1.904022,1.793014,1.334608,1.576134,1.769253,0.596505,1.964159,1.937174
Lin,1.785298,1.624845,1.361773,1.742769,1.420590,1.781017,1.359638,1.571740,0.000000,1.593641,1.655534,1.770712,1.818085,1.667738,1.665266,1.722784,1.691831
aa,1.907185,1.922033,1.748559,1.661456,1.603184,1.591543,1.298943,1.904022,1.593641,0.000000,1.777678,1.868645,1.910366,1.719725,1.925881,1.777416,1.316045


## Load Remy's updated megamat17 KC distances

For `panel_name='megamat_new_and_old_by_fly_17'`, make the following dataframes:
- df_stim_rdm_combo_mean: column name `'kc_remy_combo'`
- df_stim_rdm_combo_mean_scaled: column name `'kc_remy_combo_mean_scaled'`
- df_stim_rdm_combo_scaled_mean: column name `'kc_remy_combo_scaled_mean'`

In [81]:
panel_name = 'megamat_new_and_old_by_fly_17'

ds_stim_rdm_concat_mm_combo = xr.load_dataset(f"/home/remy/PycharmProjects/OdorSpaceShare/"
                                              f"manuscript/data/by_imaging_panel"
                                              f"/{panel_name}/"
                                              "kc_soma_nls/"
                                              "xrds_stim_rdm_concat_mm.nc"
                                              )

ds_stim_rdm_concat_mm_combo = (ds_stim_rdm_concat_mm_combo
                               .set_index(acq=['date_imaged', 'fly_num', 'thorimage_name'])
                               .set_index(rdms=['cell_mask_coord', 'metric'])
                               )

da_stim_rdm_combo = (ds_stim_rdm_concat_mm_combo['Fc_zscore'].sel(cell_mask_coord='iscell_good_xid0',
                                                                  metric='correlation'))

# compute "raw" mean correlation distances
da_stim_rdm_combo_mean = da_stim_rdm_combo.mean(dim='acq')
df_stim_rdm_combo_mean = da_stim_rdm_combo_mean.to_pandas()

# compute scaled distances (mean, then scaled)
da_stim_rdm_combo_mean_scaled = (2.0 / da_stim_rdm_combo_mean.max(dim=['stim_row', 'stim_col'])) * da_stim_rdm_combo_mean
df_stim_rdm_combo_mean_scaled = da_stim_rdm_combo_mean_scaled.to_pandas()

# compute scaled distances (scaled, then mean)
da_stim_rdm_combo_scaled = (2.0 / da_stim_rdm_combo.max(dim=['stim_row', 'stim_col'])) * da_stim_rdm_combo
da_stim_rdm_combo_scaled_mean = da_stim_rdm_scaled.mean(dim='acq')
df_stim_rdm_combo_scaled_mean = da_stim_rdm_combo_scaled_mean.to_pandas()

# Add combo dists to `neural_dists`
neural_dists['kc_remy_combo'] = df_stim_rdm_combo_mean
neural_dists['kc_remy_combo_mean_scaled'] = df_stim_rdm_combo_mean_scaled
neural_dists['kc_remy_combo_scaled_mean'] = df_stim_rdm_combo_scaled_mean

print('megamat17_new_and_old_by_fly (combined flies): `kc_remy_combo`')
display(neural_dists['kc_remy_combo'])

print('megamat17_new_and_old_by_fly (combined flies): `kc_remy_combo_mean_scaled`')
display(neural_dists['kc_remy_combo_mean_scaled'])

print('megamat17_new_and_old_by_fly (combined flies): `kc_remy_combo_scaled_mean`')
display(neural_dists['kc_remy_combo_scaled_mean'])


megamat17_new_and_old_by_fly (combined flies): `kc_remy_combo`


stim_col,1-5ol @ -3.0,1-6ol @ -3.0,1-8ol @ -3.0,2-but @ -3.0,2h @ -3.0,6al @ -3.0,B-cit @ -3.0,IaA @ -3.0,Lin @ -3.0,aa @ -3.0,benz @ -3.0,eb @ -3.0,ep @ -3.0,ms @ -3.0,pa @ -3.0,t2h @ -3.0,va @ -3.0
stim_row,,,,,,,,,,,,,,,,,
1-5ol @ -3.0,0.000000,0.277175,0.682328,0.880439,0.939584,0.578802,0.974556,0.980936,0.977610,1.104487,0.802469,0.844637,0.878089,1.026726,0.952181,0.797771,1.091494
1-6ol @ -3.0,0.277175,0.000000,0.683138,0.961685,0.853585,0.657938,0.937729,0.956020,0.914698,1.096710,0.916046,0.947949,0.958506,1.037186,0.818657,0.789219,1.057750
1-8ol @ -3.0,0.682328,0.683138,0.000000,0.933569,0.953467,0.795978,0.620097,1.036444,0.670207,0.944458,0.980877,0.955558,1.000394,0.993429,0.971294,0.827272,0.987171
2-but @ -3.0,0.880439,0.961685,0.933569,0.000000,0.968805,0.983160,0.944155,0.995000,0.965884,0.935588,1.055676,0.692948,0.453744,0.934322,1.026842,1.019845,1.008798
2h @ -3.0,0.939584,0.853585,0.953467,0.968805,0.000000,0.915467,0.972858,0.620548,0.811923,0.926241,0.959995,0.859110,0.981503,1.057925,0.543134,0.974713,1.007292
6al @ -3.0,0.578802,0.657938,0.795978,0.983160,0.915467,0.000000,0.914440,1.036884,1.038350,0.918652,0.739327,1.042477,1.087564,1.033079,0.976546,0.505587,0.867749
B-cit @ -3.0,0.974556,0.937729,0.620097,0.944155,0.972858,0.914440,0.000000,1.017236,0.817067,0.749993,0.933435,0.963511,0.988257,0.929468,1.017111,0.933938,0.907203
IaA @ -3.0,0.980936,0.956020,1.036444,0.995000,0.620548,1.036884,1.017236,0.000000,0.909610,1.077801,1.013608,0.769348,0.917583,0.990358,0.419201,1.107444,1.101141
Lin @ -3.0,0.977610,0.914698,0.670207,0.965884,0.811923,1.038350,0.817067,0.909610,0.000000,0.920961,0.946754,0.957020,1.007185,0.964448,0.961260,0.967264,0.977563


megamat17_new_and_old_by_fly (combined flies): `kc_remy_combo_mean_scaled`


stim_col,1-5ol @ -3.0,1-6ol @ -3.0,1-8ol @ -3.0,2-but @ -3.0,2h @ -3.0,6al @ -3.0,B-cit @ -3.0,IaA @ -3.0,Lin @ -3.0,aa @ -3.0,benz @ -3.0,eb @ -3.0,ep @ -3.0,ms @ -3.0,pa @ -3.0,t2h @ -3.0,va @ -3.0
stim_row,,,,,,,,,,,,,,,,,
1-5ol @ -3.0,0.000000,0.500566,1.232257,1.590038,1.696851,1.045292,1.760009,1.771532,1.765525,1.994659,1.449227,1.525380,1.585794,1.854227,1.719600,1.440742,1.971195
1-6ol @ -3.0,0.500566,0.000000,1.233720,1.736765,1.541540,1.188210,1.693501,1.726534,1.651908,1.980613,1.654342,1.711959,1.731024,1.873116,1.478462,1.425297,1.910254
1-8ol @ -3.0,1.232257,1.233720,0.000000,1.685989,1.721924,1.437505,1.119871,1.871776,1.210367,1.705652,1.771425,1.725698,1.806670,1.794094,1.754117,1.494019,1.782791
2-but @ -3.0,1.590038,1.736765,1.685989,0.000000,1.749623,1.775548,1.705106,1.796930,1.744347,1.689634,1.906507,1.251436,0.819444,1.687349,1.854435,1.841799,1.821849
2h @ -3.0,1.696851,1.541540,1.721924,1.749623,0.000000,1.653296,1.756943,1.120685,1.466300,1.672753,1.733712,1.551517,1.772555,1.910571,0.980878,1.760292,1.819129
6al @ -3.0,1.045292,1.188210,1.437505,1.775548,1.653296,0.000000,1.651442,1.872571,1.875218,1.659049,1.335195,1.882671,1.964097,1.865699,1.763602,0.913070,1.567120
B-cit @ -3.0,1.760009,1.693501,1.119871,1.705106,1.756943,1.651442,0.000000,1.837087,1.475591,1.354457,1.685747,1.740061,1.784753,1.678582,1.836861,1.686654,1.638372
IaA @ -3.0,1.771532,1.726534,1.871776,1.796930,1.120685,1.872571,1.837087,0.000000,1.642719,1.946465,1.830536,1.389411,1.657118,1.788546,0.757059,2.000000,1.988616
Lin @ -3.0,1.765525,1.651908,1.210367,1.744347,1.466300,1.875218,1.475591,1.642719,0.000000,1.663219,1.709800,1.728340,1.818936,1.741755,1.735996,1.746840,1.765439


megamat17_new_and_old_by_fly (combined flies): `kc_remy_combo_scaled_mean`


stim_col,1-5ol @ -3.0,1-6ol @ -3.0,1-8ol @ -3.0,2-but @ -3.0,2h @ -3.0,6al @ -3.0,B-cit @ -3.0,IaA @ -3.0,Lin @ -3.0,aa @ -3.0,benz @ -3.0,eb @ -3.0,ep @ -3.0,ms @ -3.0,pa @ -3.0,t2h @ -3.0,va @ -3.0
stim_row,,,,,,,,,,,,,,,,,
1-5ol @ -3.0,0.000000,0.476571,0.969711,1.643964,1.636903,0.948734,1.632497,1.767234,1.785298,1.907185,1.388153,1.557113,1.551742,1.777486,1.646505,1.172430,1.908381
1-6ol @ -3.0,0.476571,0.000000,0.825337,1.744128,1.536635,1.133350,1.547494,1.792762,1.624845,1.922033,1.508721,1.695807,1.673437,1.804145,1.527290,1.274090,1.908305
1-8ol @ -3.0,0.969711,0.825337,0.000000,1.592060,1.687797,1.315354,1.105537,1.829213,1.361773,1.748559,1.674579,1.633928,1.587604,1.697260,1.673421,1.374702,1.778949
2-but @ -3.0,1.643964,1.744128,1.592060,0.000000,1.700025,1.632075,1.579457,1.846604,1.742769,1.661456,1.857108,1.316508,0.785731,1.621305,1.904712,1.778232,1.677901
2h @ -3.0,1.636903,1.536635,1.687797,1.700025,0.000000,1.477155,1.709869,0.972714,1.420590,1.603184,1.673533,1.556201,1.724243,1.830850,0.785012,1.627404,1.647638
6al @ -3.0,0.948734,1.133350,1.315354,1.632075,1.477155,0.000000,1.524111,1.894282,1.781017,1.591543,1.363047,1.727335,1.804653,1.844132,1.792537,0.680782,1.515718
B-cit @ -3.0,1.632497,1.547494,1.105537,1.579457,1.709869,1.524111,0.000000,1.791499,1.359638,1.298943,1.736063,1.719017,1.710316,1.624644,1.800755,1.651953,1.579109
IaA @ -3.0,1.767234,1.792762,1.829213,1.846604,0.972714,1.894282,1.791499,0.000000,1.571740,1.904022,1.793014,1.334608,1.576134,1.769253,0.596505,1.964159,1.937174
Lin @ -3.0,1.785298,1.624845,1.361773,1.742769,1.420590,1.781017,1.359638,1.571740,0.000000,1.593641,1.655534,1.770712,1.818085,1.667738,1.665266,1.722784,1.691831


## Combine and save KC distances



In [97]:

tidy_kc_dists_abbrev = pd.concat([v.stack().rename(k) for k, v in neural_dists.items()], axis=1)
tidy_kc_dists_abbrev = tidy_kc_dists_abbrev.rename(index=lambda x: x.split(" @ ")[0]).rename_axis(index=[None, None])
display(tidy_kc_dists_abbrev)

tidy_kc_dists_inchi = tidy_kc_dists_abbrev.rename(index=anoop_abbrev_2_inchi)
display(tidy_kc_dists_inchi)

kc_remy  kc_remy_mean_scaled  kc_remy_scaled_mean  \
1-5ol 1-5ol  0.000000             0.000000             0.000000   
      1-6ol  0.276908             0.487828             0.476571   
      1-8ol  0.560735             0.987846             0.969711   
      2-but  0.948798             1.671495             1.643964   
      2h     0.946051             1.666657             1.636903   
...               ...                  ...                  ...   
va    ep     1.043903             1.839042             1.806334   
      ms     0.996128             1.754878             1.723054   
      pa     1.096582             1.931847             1.897909   
      t2h    0.806864             1.421452             1.397377   
      va     0.000000             0.000000             0.000000   

             kc_remy_combo  kc_remy_combo_mean_scaled  \
1-5ol 1-5ol       0.000000                   0.000000   
      1-6ol       0.277175                   0.500566   
      1-8ol       0.682328                   1.232257   
      2-but       0.880439                   1.590038   
      2h          0.939584                   1.696851   
...                    ...                        ...   
va    ep          1.059619                   1.913629   
      ms          1.016737                   1.836187   
      pa          1.041864                   1.881564   
      t2h         0.774902                   1.399442   
      va          0.000000                   0.000000   

             kc_remy_combo_scaled_mean  
1-5ol 1-5ol                   0.000000  
      1-6ol                   0.476571  
      1-8ol                   0.969711  
      2-but                   1.643964  
      2h                      1.636903  
...                                ...  
va    ep                      1.806334  
      ms                      1.723054  
      pa                      1.897909  
      t2h                     1.397377  
      va                      0.000000  

[289 rows x 6 columns]

kc_remy  \
InChI=1S/C5H12O/c1-2-3-4-5-6/h6H,2-5H2,1H3         InChI=1S/C5H12O/c1-2-3-4-5-6/h6H,2-5H2,1H3          0.000000   
                                                   InChI=1S/C6H14O/c1-2-3-4-5-6-7/h7H,2-6H2,1H3        0.276908   
                                                   InChI=1S/C8H18O/c1-2-3-4-5-6-7-8-9/h9H,2-8H2,1H3    0.560735   
                                                   InChI=1S/C4H8O/c1-3-4(2)5/h3H2,1-2H3                0.948798   
                                                   InChI=1S/C7H14O/c1-3-4-5-6-7(2)8/h3-6H2,1-2H3       0.946051   
...                                                                                                         ...   
InChI=1S/C5H10O2/c1-2-3-4-5(6)7/h2-4H2,1H3,(H,6,7) InChI=1S/C5H10O2/c1-3-5(6)7-4-2/h3-4H2,1-2H3        1.043903   
                                                   InChI=1S/C8H8O3/c1-11-8(10)6-4-2-3-5-7(6)9/h2-5...  0.996128   
                                                   InChI=1S/C7H14O2/c1-3-4-5-6-9-7(2)8/h3-6H2,1-2H3    1.096582   
                                                   InChI=1S/C6H10O/c1-2-3-4-5-6-7/h4-6H,2-3H2,1H3/...  0.806864   
                                                   InChI=1S/C5H10O2/c1-2-3-4-5(6)7/h2-4H2,1H3,(H,6,7)  0.000000   

                                                                                                       kc_remy_mean_scaled  \
InChI=1S/C5H12O/c1-2-3-4-5-6/h6H,2-5H2,1H3         InChI=1S/C5H12O/c1-2-3-4-5-6/h6H,2-5H2,1H3                     0.000000   
                                                   InChI=1S/C6H14O/c1-2-3-4-5-6-7/h7H,2-6H2,1H3                   0.487828   
                                                   InChI=1S/C8H18O/c1-2-3-4-5-6-7-8-9/h9H,2-8H2,1H3               0.987846   
                                                   InChI=1S/C4H8O/c1-3-4(2)5/h3H2,1-2H3                           1.671495   
                                                   InChI=1S/C7H14O/c1-3-4-5-6-7(2)8/h3-6H2,1-2H3                  1.666657   
...                                                                                                                    ...   
InChI=1S/C5H10O2/c1-2-3-4-5(6)7/h2-4H2,1H3,(H,6,7) InChI=1S/C5H10O2/c1-3-5(6)7-4-2/h3-4H2,1-2H3                   1.839042   
                                                   InChI=1S/C8H8O3/c1-11-8(10)6-4-2-3-5-7(6)9/h2-5...             1.754878   
                                                   InChI=1S/C7H14O2/c1-3-4-5-6-9-7(2)8/h3-6H2,1-2H3               1.931847   
                                                   InChI=1S/C6H10O/c1-2-3-4-5-6-7/h4-6H,2-3H2,1H3/...             1.421452   
                                                   InChI=1S/C5H10O2/c1-2-3-4-5(6)7/h2-4H2,1H3,(H,6,7)             0.000000   

                                                                                                       kc_remy_scaled_mean  \
InChI=1S/C5H12O/c1-2-3-4-5-6/h6H,2-5H2,1H3         InChI=1S/C5H12O/c1-2-3-4-5-6/h6H,2-5H2,1H3                     0.000000   
                                                   InChI=1S/C6H14O/c1-2-3-4-5-6-7/h7H,2-6H2,1H3                   0.476571   
                                                   InChI=1S/C8H18O/c1-2-3-4-5-6-7-8-9/h9H,2-8H2,1H3               0.969711   
                                                   InChI=1S/C4H8O/c1-3-4(2)5/h3H2,1-2H3                           1.643964   
                                                   InChI=1S/C7H14O/c1-3-4-5-6-7(2)8/h3-6H2,1-2H3                  1.636903   
...                                                                                                                    ...   
InChI=1S/C5H10O2/c1-2-3-4-5(6)7/h2-4H2,1H3,(H,6,7) InChI=1S/C5H10O2/c1-3-5(6)7-4-2/h3-4H2,1-2H3                   1.806334   
                                                   InChI=1S/C8H8O3/c1-11-8(10)6-4-2-3-5-7(6)9/h2-5...             1.723054   
                                                   InChI=1S/C7H14O2/c1-3-4-5

In [98]:
# Save to data/manuscript/figure-04/04cde/neural_remy

tidy_kc_dists_abbrev.to_pickle(data_folder.joinpath('neural_remy', 'tidy_kc_dists_abbrev.pkl'))
tidy_kc_dists_abbrev.to_csv(data_folder.joinpath('neural_remy', 'tidy_kc_dists_abbrev.tsv'), sep='\t')
print("`tidy_kc_dists_abbrev` saved.")

tidy_kc_dists_inchi.to_pickle(data_folder.joinpath('neural_remy', 'tidy_kc_dists_inchi.pkl'))
tidy_kc_dists_inchi.to_csv(data_folder.joinpath('neural_remy', 'tidy_kc_dists_inchi.tsv'), sep='\t')
print("`tidy_kc_dists_inchi` saved.")

`tidy_kc_dists_abbrev` saved.
`tidy_kc_dists_inchi` saved.
